# CAP – Globaler Lauf + Enriched Dataset

Dieses Notebook führt **Schritt 1 (CAP)** der Pipeline aus.

**Ziel:**
- Für **jede Heuristik** wird mit einem LLM eine **Core-Idea (CAP)** extrahiert.
- Die CAP-Ergebnisse werden gespeichert als CSV und PKL.
- Anschließend wird das ursprüngliche Heuristik-Dataset um die CAP-Spalten erweitert
  → **Enriched Dataset** als Basis für PPP.

**Ablauf (Zellen 1–7):**
1. Setup & Pfade
2. Imports
3. Laden des Heuristik-Datensatzes (+ optionale Filter)
4. Laden & Rendern des CAP-Prompts
5. Globaler CAP-Run (mit Resume)
6. Merge der CAP-Ergebnisse ins Original-Dataset
7. Sanity Checks

**Outputs:**
- `cap_results.csv / .pkl`
- `all_heuristics_with_cap.csv / .pkl`


In [5]:
from pathlib import Path
import sys
sys.executable

REPO_ROOT = Path.cwd().parents[1]  # notebooks/Emirhan/... -> repo root
DATA_PATH_PKL = REPO_ROOT / "data" / "all_heuristics_dataset.pkl"
DATA_PATH_CSV = REPO_ROOT / "data" / "all_heuristics_dataset.csv"

OUT_DIR = REPO_ROOT / "data" / "outputs" / "cap_global"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAP_TEMPLATE_PATH = REPO_ROOT / "src" / "prompt_templates" / "cap.md"  # adjust if needed

print("REPO_ROOT:", REPO_ROOT)
print("DATA_PATH_PKL exists:", DATA_PATH_PKL.exists(), DATA_PATH_PKL)
print("DATA_PATH_CSV exists:", DATA_PATH_CSV.exists(), DATA_PATH_CSV)
print("CAP_TEMPLATE_PATH exists:", CAP_TEMPLATE_PATH.exists(), CAP_TEMPLATE_PATH)
print("OUT_DIR:", OUT_DIR)

# make repo importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("sys.path[0]:", sys.path[0])


REPO_ROOT: /Users/emirhangunes/VSCode/ppp-performance-prediction
DATA_PATH_PKL exists: True /Users/emirhangunes/VSCode/ppp-performance-prediction/data/all_heuristics_dataset.pkl
DATA_PATH_CSV exists: True /Users/emirhangunes/VSCode/ppp-performance-prediction/data/all_heuristics_dataset.csv
CAP_TEMPLATE_PATH exists: True /Users/emirhangunes/VSCode/ppp-performance-prediction/src/prompt_templates/cap.md
OUT_DIR: /Users/emirhangunes/VSCode/ppp-performance-prediction/data/outputs/cap_global
sys.path[0]: /Users/emirhangunes/VSCode/ppp-performance-prediction


In [15]:
import pandas as pd
from tqdm.auto import tqdm

from src.api.mock_client import MockLLMClient  # swap later for real client
from src.api.parse import parse_cap_response
from src.api.deepseek_config import load_deepseek_config
from src.api.deepseek_client import DeepSeekLLMClient

import os

os.environ["DEEPSEEK_API_KEY"] = "sk-19d82747043c4fa59494bc25d816f4fe"
os.environ["DEEPSEEK_MODEL"] = "deepseek-chat"
os.environ["DEEPSEEK_BASE_URL"] = "https://api.deepseek.com"


In [7]:
# Load dataset
if DATA_PATH_PKL.exists():
    df = pd.read_pickle(DATA_PATH_PKL)
elif DATA_PATH_CSV.exists():
    df = pd.read_csv(DATA_PATH_CSV)
else:
    raise FileNotFoundError("No dataset found (.pkl or .csv).")

print("Loaded rows:", len(df))
print("Columns:", list(df.columns))

required = {"heuristic_id", "raw_app_type", "code"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# OPTIONAL: reproduce same filters you used before (recommended)
def to_bool(x):
    if isinstance(x, bool): return x
    if isinstance(x, str): return x.strip().lower() == "true"
    return False

if "parse_ok" in df.columns:
    df["parse_ok_bool"] = df["parse_ok"].apply(to_bool)
else:
    df["parse_ok_bool"] = True

if "is_timeout" in df.columns:
    df["is_timeout_bool"] = df["is_timeout"].apply(to_bool)
else:
    df["is_timeout_bool"] = False

# Decide: keep only parse_ok & not timeout
USE_FILTERS = True
if USE_FILTERS:
    df_in = df[(df["parse_ok_bool"] == True) & (df["is_timeout_bool"] == False)].copy()
else:
    df_in = df.copy()

df_in = df_in.reset_index(drop=True)
print("Using rows for CAP:", len(df_in))
df_in.head(3)[["heuristic_id","raw_app_type"]]


Loaded rows: 15507
Columns: ['heuristic_id', 'raw_app_type', 'instance_scale', 'filename', 'strategy', 'algorithm', 'code', 'objective', 'task_name', 'parse_ok', 'is_timeout']
Using rows for CAP: 15507


,heuristic_id,raw_app_type
0,pop_0_op_e1_n0_251224_134701,bin_greedy
1,pop_0_op_e1_n10_251224_134701,bin_greedy
2,pop_0_op_e1_n11_251224_134701,bin_greedy


In [8]:
cap_template = CAP_TEMPLATE_PATH.read_text(encoding="utf-8")
print("CAP template length:", len(cap_template))

def render_cap_prompt(template: str, code: str) -> str:
    # IMPORTANT: avoid .format() because template contains JSON braces
    return template.replace("{code}", code)


CAP template length: 753


In [12]:
import os
key = os.getenv("DEEPSEEK_API_KEY")
print("Key present:", key is not None, "len:", None if key is None else len(key))
print("Base URL:", os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"))
print("Model:", os.getenv("DEEPSEEK_MODEL", "deepseek-chat"))


Key present: True len: 13
Base URL: https://api.deepseek.com
Model: deepseek-chat


In [16]:
import time
import json
import os
CAP_CSV = OUT_DIR / "cap_results.csv"
CAP_PKL = OUT_DIR / "cap_results.pkl"

ENRICHED_CSV = OUT_DIR / "all_heuristics_with_cap.csv"
ENRICHED_PKL = OUT_DIR / "all_heuristics_with_cap.pkl"

# Instantiate client (Mock for now)
cfg = load_deepseek_config()
client = DeepSeekLLMClient(cfg)


#client = MockLLMClient()

# Resume support: load existing CAP results if present
if CAP_PKL.exists():
    cap_prev = pd.read_pickle(CAP_PKL)
    done_ids = set(cap_prev["heuristic_id"].astype(str).tolist())
    results = cap_prev.to_dict(orient="records")
    print(f"Resume: loaded {len(cap_prev)} existing CAP rows from {CAP_PKL}")
else:
    done_ids = set()
    results = []
    print("Resume: no previous CAP PKL found, starting fresh.")

# Run settings
TEMPERATURE = 0.2
MAX_TOKENS = 512

# Iterate
t0 = time.time()
for _, row in tqdm(df_in.iterrows(), total=len(df_in), desc="CAP (global)"):
    hid = str(row["heuristic_id"])
    if hid in done_ids:
        continue

    code = row["code"]
    prompt = render_cap_prompt(cap_template, str(code))

    resp = client.generate(
        prompt,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        stop=None,
        meta={"stage": "cap", "heuristic_id": hid, "raw_app_type": str(row.get("raw_app_type", ""))},
    )

    core_idea, conf, ok, err, extra = parse_cap_response(resp.text)

    results.append({
        "heuristic_id": hid,
        "raw_app_type": str(row.get("raw_app_type", "")),
        "cap_core_idea": core_idea,
        "cap_confidence": conf,
        "cap_parse_ok": bool(ok),
        "cap_parse_error": err,
        "cap_model": resp.model,
        "cap_prompt_hash": resp.prompt_hash,
        "cap_latency_s": float(resp.latency_s),
        "cap_cached": bool(resp.cached),
        # store raw only if you want; can get huge:
        # "cap_raw_text": resp.text,
    })

    done_ids.add(hid)

    # periodic checkpoint
    if len(results) % 200 == 0:
        cap_df = pd.DataFrame(results)
        cap_df.to_pickle(CAP_PKL)
        cap_df.to_csv(CAP_CSV, index=False)
        print(f"[checkpoint] saved {len(cap_df)} rows")

elapsed = time.time() - t0
print(f"CAP finished. Rows: {len(results)}. Time: {elapsed:.1f}s")

cap_df = pd.DataFrame(results)
cap_df.to_pickle(CAP_PKL)
cap_df.to_csv(CAP_CSV, index=False)

print("Saved:", CAP_CSV)
print("Saved:", CAP_PKL)

cap_df.head(5)


Resume: no previous CAP PKL found, starting fresh.


CAP (global):   1%|▏         | 200/15507 [11:23<13:58:51,  3.29s/it]

[checkpoint] saved 200 rows


CAP (global):   3%|▎         | 400/15507 [23:27<13:54:41,  3.32s/it]

[checkpoint] saved 400 rows


...

CAP (global):  99%|█████████▉| 15400/15507 [16:27:52<04:58,  2.79s/it]

[checkpoint] saved 15400 rows


CAP (global): 100%|██████████| 15507/15507 [16:34:12<00:00,  3.85s/it]

CAP finished. Rows: 15507. Time: 59652.4s
Saved: /Users/emirhangunes/VSCode/ppp-performance-prediction/data/outputs/cap_global/cap_results.csv
Saved: /Users/emirhangunes/VSCode/ppp-performance-prediction/data/outputs/cap_global/cap_results.pkl


,heuristic_id,raw_app_type,cap_core_idea,cap_confidence,cap_parse_ok,cap_parse_error,cap_model,cap_prompt_hash,cap_latency_s,cap_cached
0,pop_0_op_e1_n0_251224_134701,bin_greedy,The heuristic is a scoring function for bin pa...,None,True,None,deepseek-chat,8a7b9b577c3452858ca5f47bbb88c646c1467be4d5f1d7...,3.614690,True
1,pop_0_op_e1_n10_251224_134701,bin_greedy,The heuristic evaluates candidate placements b...,None,True,None,deepseek-chat,19e6c750cfb0b9eb898923973a972ce3d369070d2d6886...,2.659119,False
2,pop_0_op_e1_n11_251224_134701,bin_greedy,The heuristic selects a placement for an item ...,None,True,None,deepseek-chat,f559c243bdb0acad0a3b1dee7af3b4a1da7bf4ee8b5e71...,3.170817,False
3,pop_0_op_e1_n12_251224_134701,bin_greedy,The heuristic selects a placement for an item ...,None,True,None,deepseek-chat,0dffda822db831d15aea06363f1d28d7201be4f89b6713...,3.358858,False
4,pop_0_op_e1_n13_251224_134701,bin_greedy,The heuristic evaluates candidate placements b...,None,True,None,deepseek-chat,3591ffc2e4ad568fa64161bb3989a78a51c489e5e38ea5...,3.093505,False


In [17]:
# Merge CAP results into the ORIGINAL df (not df_in), so you keep all columns
cap_df = pd.read_pickle(CAP_PKL)
cap_df = cap_df.drop_duplicates(subset=["heuristic_id"], keep="last").copy()

df_all = df.copy()
df_all["heuristic_id"] = df_all["heuristic_id"].astype(str)
cap_df["heuristic_id"] = cap_df["heuristic_id"].astype(str)

df_enriched = df_all.merge(
    cap_df[["heuristic_id", "cap_core_idea", "cap_confidence", "cap_parse_ok", "cap_parse_error"]],
    on="heuristic_id",
    how="left"
)

print("Enriched rows:", len(df_enriched))
print("CAP coverage (non-null core ideas):", df_enriched["cap_core_idea"].notna().sum())

# Save enriched
df_enriched.to_pickle(ENRICHED_PKL)
df_enriched.to_csv(ENRICHED_CSV, index=False)

print("Saved:", ENRICHED_PKL)
print("Saved:", ENRICHED_CSV)

df_enriched.head(3)[["heuristic_id","raw_app_type","cap_parse_ok","cap_core_idea"]]


Enriched rows: 15507
CAP coverage (non-null core ideas): 15507
Saved: /Users/emirhangunes/VSCode/ppp-performance-prediction/data/outputs/cap_global/all_heuristics_with_cap.pkl
Saved: /Users/emirhangunes/VSCode/ppp-performance-prediction/data/outputs/cap_global/all_heuristics_with_cap.csv


,heuristic_id,raw_app_type,cap_parse_ok,cap_core_idea
0,pop_0_op_e1_n0_251224_134701,bin_greedy,True,The heuristic is a scoring function for bin pa...
1,pop_0_op_e1_n10_251224_134701,bin_greedy,True,The heuristic evaluates candidate placements b...
2,pop_0_op_e1_n11_251224_134701,bin_greedy,True,The heuristic selects a placement for an item ...


# Nächster Schritt: PPP (Prediction)

Ab hier ist **CAP abgeschlossen**.

Als Input für PPP wird verwendet:
- `all_heuristics_with_cap.pkl`

In Schritt 2 (PPP):
- Für jede Heuristik wird ein Objective vorhergesagt
- Referenzen werden pro Task (best / mid / worst) gewählt
- Predictions sind auf den **Task-globalen Objective-Bereich** begrenzt (Variante A)

Danach erfolgt die Evaluation **pro Task**.


In [18]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parents[1]

ENRICHED_PKL = REPO_ROOT / "data" / "outputs" / "cap_global" / "all_heuristics_with_cap.pkl"
OUT_DIR = REPO_ROOT / "data" / "outputs" / "ppp_global"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PPP_TEMPLATE_PATH = REPO_ROOT / "src" / "prompt_templates" / "ppp_with_refs.md"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("ENRICHED_PKL exists:", ENRICHED_PKL.exists())
print("PPP_TEMPLATE_PATH exists:", PPP_TEMPLATE_PATH.exists())
print("OUT_DIR:", OUT_DIR)


ENRICHED_PKL exists: True
PPP_TEMPLATE_PATH exists: True
OUT_DIR: /Users/emirhangunes/VSCode/ppp-performance-prediction/data/outputs/ppp_global


In [19]:
import pandas as pd
from tqdm.auto import tqdm

from src.api.parse import parse_ppp_response


In [25]:
df = pd.read_pickle(ENRICHED_PKL)
print("Rows:", len(df))
print("Columns:", list(df.columns))

required = {
    "heuristic_id",
    "raw_app_type",
    "objective",
    "cap_core_idea",
    "cap_parse_ok"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[df["cap_parse_ok"] == True].copy()
df = df.dropna(subset=["objective", "cap_core_idea"]).reset_index(drop=True)

print("Usable rows:", len(df))


Rows: 15507
Columns: ['heuristic_id', 'raw_app_type', 'instance_scale', 'filename', 'strategy', 'algorithm', 'code', 'objective', 'task_name', 'parse_ok', 'is_timeout', 'parse_ok_bool', 'is_timeout_bool', 'cap_core_idea', 'cap_confidence', 'cap_parse_ok', 'cap_parse_error']
Usable rows: 15507


In [26]:
import os
from src.api.deepseek_config import load_deepseek_config
from src.api.deepseek_client import DeepSeekLLMClient

# (Optional) falls du den Key hier setzen willst:
# os.environ["DEEPSEEK_API_KEY"] = "PASTE_KEY_HERE".strip()

cfg = load_deepseek_config()
client = DeepSeekLLMClient(cfg)

# Preflight: bricht sofort, wenn Auth/Endpoint kaputt ist
resp = client.generate('Return JSON only: {"ping":"pong"}', temperature=0.0, max_tokens=30, stop=None, meta={"stage":"test"})
print("Preflight OK. First chars:", resp.text[:80])


Preflight OK. First chars: ```json
{"ping":"pong"}
```


In [27]:
def select_refs_stratified(df_task, target_id, k=3):
    df_cand = df_task[df_task["heuristic_id"] != target_id].copy()
    df_cand = df_cand.sort_values("objective", ascending=True)

    if len(df_cand) < k:
        return None

    best = df_cand.iloc[0]
    worst = df_cand.iloc[-1]
    mid = df_cand.iloc[len(df_cand)//2]

    return [best, mid, worst]


def build_refs_block(refs):
    lines = []
    for i, r in enumerate(refs, 1):
        lines += [
            f"{i})",
            "Core Idea:",
            r["cap_core_idea"],
            f"Objective: {r['objective']}",
            ""
        ]
    return "\n".join(lines).strip() + "\n"


ppp_template = PPP_TEMPLATE_PATH.read_text(encoding="utf-8")

def render_ppp_prompt(**kwargs):
    return ppp_template.format(**kwargs)


In [28]:
from src.api.parse import parse_ppp_response

test_prompt = render_ppp_prompt(
    raw_app_type="TEST",
    task_min=0.0,
    task_max=1.0,
    references_block=(
        "1)\nCore Idea:\nExample A\nObjective: 0.1\n\n"
        "2)\nCore Idea:\nExample B\nObjective: 0.5\n\n"
        "3)\nCore Idea:\nExample C\nObjective: 0.9\n"
    ),
    target_core="Target idea"
)

resp = client.generate(test_prompt, temperature=0.2, max_tokens=256, stop=None, meta={"stage":"ppp","heuristic_id":"TEST"})
print("Raw response:\n", resp.text)

pred, conf, ok, err, extra = parse_ppp_response(resp.text)
print("parse_ok:", ok, "pred:", pred, "conf:", conf, "err:", err)


Raw response:
 ```json
{
  "prediction": 0.5,
  "confidence": 0.3,
  "justification": "The target idea is too generic to determine clear algorithmic similarity to any specific reference (Example A, B, or C). Given the lack of distinguishing features, the prediction defaults to the middle of the reference range (0.5), which corresponds to Example B's value, but with low confidence due to insufficient semantic detail for meaningful comparison."
}
```
parse_ok: True pred: 0.5 conf: 0.3 err: None


In [29]:
task_bounds = (
    df.groupby("raw_app_type")["objective"]
      .agg(["min", "max"])
      .to_dict(orient="index")
)

task_bounds


{'bin_greedy': {'min': 0.00563, 'max': 1.51534},
 'cvrp_lns': {'min': 3343425.3, 'max': 3981765.6},
 'premarshalling_astar': {'min': 0.08148, 'max': 13.9329},
 'puzzle_astar': {'min': 0.4574, 'max': 3.65888}}

In [30]:
TEST_PER_TASK = 3

for app, df_task in df.groupby("raw_app_type"):
    task_min = task_bounds[app]["min"]
    task_max = task_bounds[app]["max"]

    print("\n=== APP:", app, "| rows:", len(df_task), "| range:", (task_min, task_max))

    df_task_small = df_task.head(TEST_PER_TASK)

    for _, row in df_task_small.iterrows():
        hid = str(row["heuristic_id"])

        refs = select_refs_stratified(df_task, hid, k=3)
        if refs is None:
            print("No refs for", hid)
            continue

        prompt = render_ppp_prompt(
            raw_app_type=app,
            task_min=task_min,
            task_max=task_max,
            references_block=build_refs_block(refs),
            target_core=row["cap_core_idea"],
        )

        resp = client.generate(prompt, temperature=0.2, max_tokens=512, stop=None, meta={"stage":"ppp","heuristic_id":hid})
        pred, conf, ok, err, _ = parse_ppp_response(resp.text)

        if pred is not None:
            pred = max(task_min, min(task_max, pred))

        print("hid:", hid, "| obj:", row["objective"], "| pred:", pred, "| conf:", conf, "| ok:", ok, "| err:", err)



=== APP: bin_greedy | rows: 4445 | range: (0.00563, 1.51534)
hid: pop_0_op_e1_n0_251224_134701 | obj: 1.51534 | pred: 0.05071 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n10_251224_134701 | obj: 0.3277 | pred: 0.05071 | conf: 0.75 | ok: True | err: None
hid: pop_0_op_e1_n11_251224_134701 | obj: 1.51534 | pred: 0.02817 | conf: 0.65 | ok: True | err: None

=== APP: cvrp_lns | rows: 1557 | range: (3343425.3, 3981765.6)
hid: pop_0_op_e1_n0_251225_153652 | obj: 3978416.0 | pred: 3560000.0 | conf: 0.7 | ok: True | err: None
hid: pop_0_op_e1_n11_251225_153652 | obj: 3613762.5 | pred: 3692252.8 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n12_251225_153652 | obj: 3956702.6 | pred: 3650000.0 | conf: 0.75 | ok: True | err: None

=== APP: premarshalling_astar | rows: 4647 | range: (0.08148, 13.9329)
hid: pop_0_op_e1_n0_250815_114538 | obj: 10.69957 | pred: 0.57173 | conf: 0.65 | ok: True | err: None
hid: pop_0_op_e1_n10_250815_114538 | obj: 8.79957 | pred: 8.0 | conf: 0.65 | ok:

In [32]:
PPP_PKL = OUT_DIR / "ppp_results.pkl"
PPP_CSV = OUT_DIR / "ppp_results.csv"



if PPP_PKL.exists():
    prev = pd.read_pickle(PPP_PKL)
    done_ids = set(prev["heuristic_id"].astype(str))
    results = prev.to_dict(orient="records")
    print("Resuming, loaded:", len(done_ids))
else:
    done_ids = set()
    results = []

for app, df_task in df.groupby("raw_app_type"):
    task_min = task_bounds[app]["min"]
    task_max = task_bounds[app]["max"]

    for _, row in tqdm(df_task.iterrows(), total=len(df_task), desc=f"PPP [{app}]"):
        hid = str(row["heuristic_id"])
        if hid in done_ids:
            continue

        refs = select_refs_stratified(df_task, hid, k=3)
        if refs is None:
            continue

        prompt = render_ppp_prompt(
            raw_app_type=app,
            task_min=task_min,
            task_max=task_max,
            references_block=build_refs_block(refs),
            target_core=row["cap_core_idea"],
        )

        resp = client.generate(
            prompt,
            temperature=0.2,
            max_tokens=512,
            stop=None,
            meta={"stage": "ppp", "heuristic_id": hid}
        )

        pred, conf, ok, err, _ = parse_ppp_response(resp.text)

        if pred is not None:
            pred = max(task_min, min(task_max, pred))  # clamp (Variante A)

        results.append({
            "heuristic_id": hid,
            "raw_app_type": app,
            "objective": row["objective"],
            "prediction": pred,
            "confidence": conf,
            "parse_ok": ok,
            "parse_error": err,
        })

        done_ids.add(hid)
        
        # periodic checkpoint
        if len(results) % 200 == 0:
            ppp_df = pd.DataFrame(results)
            ppp_df.to_pickle(PPP_PKL)
            ppp_df.to_csv(PPP_CSV, index=False)
            print("[checkpoint] saved", len(results))

# save
ppp_df = pd.DataFrame(results)
ppp_df.to_pickle(PPP_PKL)
ppp_df.to_csv(PPP_CSV, index=False)

print("Saved:", PPP_PKL)
print("Saved:", PPP_CSV)


PPP [bin_greedy]:   0%|          | 18/4445 [01:27<6:00:36,  4.89s/it]


KeyboardInterrupt: 